# Multi-Part Car Damage & Severity Classification

## 1. Setup

In [ ]:
from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/cardamage6")
DRIVE_RUNS = DRIVE_ROOT / "runs"
LOCAL_DATA = Path("/content/data")

for p in (DRIVE_RUNS, LOCAL_DATA):
    p.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "GPU runtime not active."
print(f"PyTorch {torch.__version__} | Device: {torch.cuda.get_device_name(0)}")

In [ ]:
import os

REPO_URL = "https://github.com/deba75/Car-Damage.git"
REPO_DIR = "/content/Car-Damage"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q

%cd {REPO_DIR}
!pip install -q "timm>=0.9" "pycocotools>=2.0.7" "ultralytics>=8.1" kagglehub

In [ ]:
!python smoke_test.py

## 2. Fetch Kaggle Dataset (Cloud-to-Cloud)

In [ ]:
import kagglehub

KAGGLE_DIR = Path(kagglehub.dataset_download("lplenka/coco-car-damage-detection-dataset"))
print("Dataset cached at:", KAGGLE_DIR)
!find "{KAGGLE_DIR}" -maxdepth 2

## 3. Build & Merge Labels

In [ ]:
train_ann = next(KAGGLE_DIR.rglob("COCO_train_annos.json"))
train_parts = next(KAGGLE_DIR.rglob("COCO_mul_train_annos.json"))
val_ann = next(KAGGLE_DIR.rglob("COCO_val_annos.json"))
val_parts = next(KAGGLE_DIR.rglob("COCO_mul_val_annos.json"))
img_dirs = [p for p in KAGGLE_DIR.rglob("img") if p.is_dir()]
img_root = img_dirs[0] if img_dirs else train_ann.parent

PAIRED_COCO_DATASETS = [
    {"source": "lplenka", "images": str(img_root), "ann": str(train_ann), "parts_ann": str(train_parts), "out": str(LOCAL_DATA / "labels_lplenka_train.csv")},
    {"source": "lplenka", "images": str(img_root), "ann": str(val_ann),   "parts_ann": str(val_parts),   "out": str(LOCAL_DATA / "labels_lplenka_val.csv")},
]

for ds in PAIRED_COCO_DATASETS:
    !python build_labels.py coco --images "{ds['images']}" --ann "{ds['ann']}" --parts-ann "{ds['parts_ann']}" --source "{ds['source']}" --out "{ds['out']}"

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from taxonomy import PARTS

LABEL_CSVS = [ds["out"] for ds in PAIRED_COCO_DATASETS]
SYNTHETIC_SOURCES = {"syndcar", "crashcar101"}
MERGED_CSV = LOCAL_DATA / "labels.csv"
SEED = 0

df = pd.concat([pd.read_csv(p) for p in LABEL_CSVS], ignore_index=True)
df = df[df["path"].map(lambda p: Path(p).exists())].drop_duplicates(subset=["path"]).reset_index(drop=True)
df["group"] = df["source"].astype(str) + ":" + df["path"].map(lambda p: Path(p).stem)

is_syn = df["source"].str.lower().isin(SYNTHETIC_SOURCES)
real_df, syn_df = df[~is_syn].copy(), df[is_syn].copy()

split_target = real_df if len(real_df) >= 3 else df.copy()
tr_idx, rest = next(GroupShuffleSplit(1, train_size=0.70, random_state=SEED).split(split_target, groups=split_target["group"]))
va_idx, te_idx = next(GroupShuffleSplit(1, train_size=0.50, random_state=SEED).split(split_target.iloc[rest], groups=split_target["group"].iloc[rest]))

split_target["split"] = "train"
split_target.iloc[rest[va_idx], split_target.columns.get_loc("split")] = "val"
split_target.iloc[rest[te_idx], split_target.columns.get_loc("split")] = "test"

if len(real_df) >= 3:
    syn_df["split"] = "train"
    merged = pd.concat([split_target, syn_df], ignore_index=True)
else:
    merged = split_target

merged.to_csv(MERGED_CSV, index=False)
merged.to_csv(DRIVE_ROOT / "labels.csv", index=False)

print(merged.groupby(["split", "source"]).size().unstack(fill_value=0))
display(pd.DataFrame({p: merged[p].value_counts() for p in PARTS}).fillna(0).astype(int).T)

## 4. Train & Evaluate

In [ ]:
ARCH = "convnext_small.fb_in22k_ft_in1k"
IMG_SIZE = 384
EPOCHS = 12
BATCH_SIZE = 16
LR = 1.5e-4
LOSS = "focal"
SAMPLER = "on"
SEED = 0
OUT_DIR = DRIVE_RUNS / f"{ARCH.split('.')[0]}_{LOSS}_seed{SEED}"

!python train.py \
    --csv "{MERGED_CSV}" \
    --arch "{ARCH}" \
    --size {IMG_SIZE} \
    --epochs {EPOCHS} \
    --bs {BATCH_SIZE} \
    --lr {LR} \
    --loss {LOSS} \
    --sampler {SAMPLER} \
    --workers 2 \
    --seed {SEED} \
    --out "{OUT_DIR}"

In [ ]:
import json
import pandas as pd

metrics = json.loads((OUT_DIR / "test_metrics.json").read_text(encoding="utf-8"))
summary_keys = ["presence_macro_F1", "presence_macro_AP", "severity_macro_F1", "part_set_exact_match", "exact_match_part+severity"]
display(pd.Series({k: metrics[k] for k in summary_keys}, name="score").to_frame())
display(pd.DataFrame(metrics["parts"]).T)

## 5. Inference on Test Images

In [ ]:
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms as T
from train import Net, _load_state_dict

meta = json.loads((OUT_DIR / "meta.json").read_text(encoding="utf-8"))
dev = "cuda" if torch.cuda.is_available() else "cpu"
model = Net(meta["arch"], pretrained=False).to(dev)
model.load_state_dict(_load_state_dict(OUT_DIR / "best.pt", dev))
model.eval()

tf = T.Compose([
    T.Resize((meta["size"], meta["size"])),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

test_dirs = [p for p in KAGGLE_DIR.rglob("test") if p.is_dir()]
test_imgs = sorted([p for p in test_dirs[0].glob("*.*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]) if test_dirs else []

preds = []
with torch.no_grad():
    for p in test_imgs:
        x = tf(Image.open(p).convert("RGB"))[None].to(dev)
        pr = ((F.softmax(model(x).float(), -1) + F.softmax(model(x.flip(-1)).float(), -1)) / 2)[0].cpu().numpy()
        row = {"image": p.name}
        for j, part in enumerate(PARTS):
            damaged = (1 - pr[j, 0]) > meta["thresholds"][j]
            high = (pr[j, 2] / (pr[j, 1] + pr[j, 2] + 1e-9)) > meta.get("sev_thr", 0.5)
            row[part] = ("high" if high else "low") if damaged else "none"
        preds.append(row)

display(pd.DataFrame(preds))